# 8-Puzzle Solver — Thuật Toán Tham Lam (Greedy Best-First Search)

## 
| Tiêu chí (Greedy Best-First Search)
|---|---|---|
| Cách chọn nước đi | Chọn nước tốt nhất tại chỗ, bước qua bước | Dùng hàng đợi ưu tiên, mở rộng node tốt nhất toàn cục |
| Heuristic | Số ô sai vị trí | **Khoảng cách Manhattan** (chính xác hơn) |
| Tránh vòng lặp | Phạt điểm +100 | Tập `visited` lưu hash trạng thái |
| Đảm bảo tìm ra lời giải | Không (có thể kẹt) | Có (nếu bài toán giải được) |
| Tái tạo đường đi | Không | Có (truy vết từ đích về gốc) |

## Nguyên lý Greedy Best-First Search
Thuật toán luôn mở rộng node có **giá trị heuristic h(n) nhỏ nhất** — tức là trạng thái trông "gần đích nhất". Không tính chi phí đường đi g(n) như A*, nên nhanh hơn nhưng không đảm bảo tối ưu số bước.

In [1]:
import random
import heapq

# ─── Hằng số dùng chung ───────────────────────────────────────────────────────
GOAL = [[1, 2, 3], [4, 5, 6], [7, 8, 0]]
OPPOSITES = {'UP': 'DOWN', 'DOWN': 'UP', 'LEFT': 'RIGHT', 'RIGHT': 'LEFT'}

# ─── Tiện ích bàn cờ ──────────────────────────────────────────────────────────
def is_solvable(state):
    """Đếm số inversions để kiểm tra bài toán có giải được không."""
    inversions = sum(
        1 for i in range(9) for j in range(i + 1, 9)
        if state[i] and state[j] and state[i] > state[j]
    )
    return inversions % 2 == 0

def generate_random_board():
    """Khởi tạo bàn cờ ngẫu nhiên hợp lệ (luôn giải được)."""
    while True:
        nums = list(range(9))
        random.shuffle(nums)
        if is_solvable(nums):
            return [nums[i:i + 3] for i in range(0, 9, 3)]

def find_blank(board):
    """Tìm vị trí ô trống (số 0)."""
    for r in range(3):
        for c in range(3):
            if board[r][c] == 0:
                return r, c

def get_legal_moves(board):
    """Trả về dict {hướng: (hàng, cột)} cho các nước đi hợp lệ."""
    r, c = find_blank(board)
    moves = {}
    if r > 0: moves['UP']    = (r - 1, c)
    if r < 2: moves['DOWN']  = (r + 1, c)
    if c > 0: moves['LEFT']  = (r, c - 1)
    if c < 2: moves['RIGHT'] = (r, c + 1)
    return moves

def apply_move(board, move):
    """Thực hiện nước đi, trả về bàn cờ mới (không sửa bàn cũ)."""
    r, c = find_blank(board)
    nr, nc = get_legal_moves(board)[move]
    new_board = [row[:] for row in board]
    new_board[r][c], new_board[nr][nc] = new_board[nr][nc], new_board[r][c]
    return new_board

def board_to_tuple(board):
    """Chuyển bàn cờ sang tuple để dùng làm key trong set/dict."""
    return tuple(cell for row in board for cell in row)

def print_board(board, title=None):
    """In bàn cờ dạng lưới 3x3."""
    if title:
        print(f"\n{title}")
    for row in board:
        print(" ".join(f"[{n if n else ' '}]" for n in row))

## Heuristic: Khoảng cách Manhattan

Với mỗi ô (trừ ô trống), tính tổng `|hàng_hiện_tại - hàng_đích| + |cột_hiện_tại - cột_đích|`.

Manhattan Distance **chính xác hơn** heuristic số ô sai vị trí vì nó tính "bao xa" chứ không chỉ "có sai không".

In [2]:
# Bảng tra cứu vị trí đích của mỗi số (số → (hàng, cột))
GOAL_POSITIONS = {
    GOAL[r][c]: (r, c)
    for r in range(3) for c in range(3)
    if GOAL[r][c] != 0
}

def manhattan_distance(board):
    """
    [HEURISTIC] Tổng khoảng cách Manhattan của tất cả các ô đến vị trí đích.
    h(n) = Σ |r_hiện - r_đích| + |c_hiện - c_đích|  (với mọi ô ≠ 0)
    """
    total = 0
    for r in range(3):
        for c in range(3):
            tile = board[r][c]
            if tile != 0:
                goal_r, goal_c = GOAL_POSITIONS[tile]
                total += abs(r - goal_r) + abs(c - goal_c)
    return total

## Thuật toán Greedy Best-First Search

```
Hàng đợi ưu tiên (min-heap) theo h(n)
    ↓
Lấy node có h(n) nhỏ nhất
    ↓
Kiểm tra đích → nếu đúng, truy vết đường đi
    ↓
Sinh các trạng thái con, bỏ qua đã thăm
    ↓
Thêm vào hàng đợi với h(n) làm độ ưu tiên
```

Khác với A* (dùng `f = g + h`), Greedy chỉ dùng `f = h` — tức là hoàn toàn hướng về đích, bỏ qua chi phí đã đi.

In [3]:
def greedy_best_first_search(start_board, max_nodes=100000):
    """
    Greedy Best-First Search cho bài toán 8-puzzle.

    Mỗi phần tử trong heap: (h, counter, board, path)
      - h       : giá trị heuristic Manhattan (độ ưu tiên)
      - counter : bộ đếm tiebreak để tránh so sánh board (không hashable trực tiếp)
      - board   : trạng thái hiện tại
      - path    : danh sách các nước đi từ đầu đến đây

    Trả về: (path_moves, states_explored) hoặc (None, nodes_tried)
    """
    start_h = manhattan_distance(start_board)

    # Heap: (heuristic, id, board, path)
    heap = [(start_h, 0, start_board, [])]
    visited = {board_to_tuple(start_board)}   # Tập trạng thái đã thăm
    counter = 1                               # Tiebreaker duy nhất
    nodes_explored = 0

    while heap:
        if nodes_explored >= max_nodes:
            return None, nodes_explored       # Vượt giới hạn

        h, _, board, path = heapq.heappop(heap)
        nodes_explored += 1

        # ── Kiểm tra đích ──────────────────────────────────────────────
        if board == GOAL:
            return path, nodes_explored

        # ── Sinh các trạng thái kế tiếp ────────────────────────────────
        for move in get_legal_moves(board):
            # Chống đi lùi trực tiếp
            if path and move == OPPOSITES[path[-1]]:
                continue

            new_board = apply_move(board, move)
            key = board_to_tuple(new_board)

            if key not in visited:
                visited.add(key)
                new_h = manhattan_distance(new_board)   # f = h (Greedy)
                heapq.heappush(heap, (new_h, counter, new_board, path + [move]))
                counter += 1

    return None, nodes_explored   # Không tìm thấy

## Hàm chạy chính và hiển thị kết quả

In [4]:
def solve_puzzle():
    """Sinh bàn cờ ngẫu nhiên, giải bằng Greedy Best-First Search và hiển thị kết quả."""
    board = generate_random_board()
    print_board(board, "TRẠNG THÁI BẮT ĐẦU")
    print(f"\nHeuristic ban đầu (Manhattan): {manhattan_distance(board)}")
    print("\n" + "─" * 32)
    print("Đang tìm kiếm bằng Greedy Best-First Search...")

    path, nodes_explored = greedy_best_first_search(board)

    print("\n" + "=" * 32)
    if path is None:
        print("KẾT QUẢ: KHÔNG TÌM THẤY lời giải trong giới hạn cho phép.")
        print(f"Nodes đã duyệt: {nodes_explored}")
        return

    # ── Phát lại từng bước để minh họa ────────────────────────────────────
    current = board
    for step, move in enumerate(path, 1):
        current = apply_move(current, move)
        print_board(current, f"=> Bước {step}: Đi [{move}]  | h = {manhattan_distance(current)}")

    print("\n" + "=" * 32)
    print("KẾT QUẢ TỔNG QUAN")
    print(f"Thuật toán      : Greedy Best-First Search (heuristic = Manhattan Distance)")
    print(f"Tổng số bước    : {len(path)}")
    print(f"Nodes đã duyệt  : {nodes_explored}")
    print(f"Đường đi        : {' → '.join(path)}")
    print(f"Trạng thái      : SOLVED ✓")

# ── Khởi chạy ─────────────────────────────────────────────────────────────────
solve_puzzle()


TRẠNG THÁI BẮT ĐẦU
[8] [4] [5]
[2] [7] [1]
[6] [ ] [3]

Heuristic ban đầu (Manhattan): 19

────────────────────────────────
Đang tìm kiếm bằng Greedy Best-First Search...


=> Bước 1: Đi [UP]  | h = 18
[8] [4] [5]
[2] [ ] [1]
[6] [7] [3]

=> Bước 2: Đi [UP]  | h = 17
[8] [ ] [5]
[2] [4] [1]
[6] [7] [3]

=> Bước 3: Đi [LEFT]  | h = 16
[ ] [8] [5]
[2] [4] [1]
[6] [7] [3]

=> Bước 4: Đi [DOWN]  | h = 15
[2] [8] [5]
[ ] [4] [1]
[6] [7] [3]

=> Bước 5: Đi [RIGHT]  | h = 14
[2] [8] [5]
[4] [ ] [1]
[6] [7] [3]

=> Bước 6: Đi [UP]  | h = 13
[2] [ ] [5]
[4] [8] [1]
[6] [7] [3]

=> Bước 7: Đi [RIGHT]  | h = 12
[2] [5] [ ]
[4] [8] [1]
[6] [7] [3]

=> Bước 8: Đi [DOWN]  | h = 11
[2] [5] [1]
[4] [8] [ ]
[6] [7] [3]

=> Bước 9: Đi [DOWN]  | h = 10
[2] [5] [1]
[4] [8] [3]
[6] [7] [ ]

=> Bước 10: Đi [LEFT]  | h = 11
[2] [5] [1]
[4] [8] [3]
[6] [ ] [7]

=> Bước 11: Đi [UP]  | h = 10
[2] [5] [1]
[4] [ ] [3]
[6] [8] [7]

=> Bước 12: Đi [UP]  | h = 9
[2] [ ] [1]
[4] [5] [3]
[6] [8] [7]

=> Bước 13: Đi [

## So sánh heuristic: Ô sai vị trí vs Manhattan Distance

Ô dưới so sánh hai heuristic trên cùng một bàn cờ.

In [5]:
def misplaced_tiles(board):
    """Heuristic gốc: đếm số ô sai vị trí."""
    return sum(
        1 for r in range(3) for c in range(3)
        if board[r][c] and board[r][c] != GOAL[r][c]
    )

def compare_heuristics():
    """In bảng so sánh hai heuristic trên cùng bàn cờ."""
    board = generate_random_board()
    print_board(board, "Bàn cờ thử nghiệm")

    mt  = misplaced_tiles(board)
    mhd = manhattan_distance(board)

    print(f"\n{'Heuristic':<30} {'Giá trị':>8}")
    print("-" * 40)
    print(f"{'Số ô sai vị trí (file gốc)':<30} {mt:>8}")
    print(f"{'Manhattan Distance (file này)':<30} {mhd:>8}")
    print("\n→ Manhattan luôn ≥ Số ô sai vị trí")
    print("→ Manhattan informed hơn → dẫn đường tốt hơn cho Greedy")

compare_heuristics()


Bàn cờ thử nghiệm
[6] [2] [7]
[ ] [3] [1]
[5] [4] [8]

Heuristic                       Giá trị
----------------------------------------
Số ô sai vị trí (file gốc)            7
Manhattan Distance (file này)        17

→ Manhattan luôn ≥ Số ô sai vị trí
→ Manhattan informed hơn → dẫn đường tốt hơn cho Greedy
